# RetainIQ — Phase 6.4: Geospatial Reporting Layer & MySQL Publishing

I will take the Phase 6 analytical outputs and publish them into reusable MySQL tables and views so the results can be queried from SQL, Power BI, or later phases.

## Publishing rule

I will rebuild the reporting tables on each successful notebook run by clearing the previous rows before inserting the current Phase 6 outputs. I will also validate the published row counts against the source artifacts.

In [4]:
from getpass import getpass
from pathlib import Path
import pandas as pd
import mysql.connector
from IPython.display import display

MYSQL_CONFIG = {
    "host": "localhost",
    "port": 3306,
    "user": "retainiq_user",
    "password": getpass("Enter MySQL password for retainiq_user: "),
    "database": "retainiq",
}

def get_connection():
    return mysql.connector.connect(**MYSQL_CONFIG)

def print_result(message):
    print(f"Result: {message}")

output_dir = Path("../outputs")
required_files = [
    "state_retention_profile.csv",
    "city_retention_profile.csv",
    "market_segment_risk.csv",
    "geographic_grid_profile.csv"
]
missing_files = [name for name in required_files if not (output_dir / name).exists()]
if missing_files:
    raise FileNotFoundError(
        f"Run Phase 6.2 and 6.3 first. Missing: {missing_files}"
    )

state_profile = pd.read_csv(output_dir / "state_retention_profile.csv")
city_profile = pd.read_csv(output_dir / "city_retention_profile.csv")
market_segment_risk = pd.read_csv(output_dir / "market_segment_risk.csv")
grid_profile = pd.read_csv(output_dir / "geographic_grid_profile.csv")

print_result("I found all required Phase 6 output files and loaded them for publishing.")

Result: I found all required Phase 6 output files and loaded them for publishing.


### Result & conclusion

The reporting notebook is now working from saved Phase 6 artifacts rather than rebuilding analysis logic. This keeps the reporting layer separate from exploratory analysis.

## 1. Create the reporting tables

In [5]:
connection = get_connection()
cursor = connection.cursor()

try:
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS state_retention_profile (
            state VARCHAR(100) NOT NULL,
            customers INT NOT NULL,
            churned_customers INT NOT NULL,
            avg_cltv DECIMAL(14,2),
            total_revenue DECIMAL(16,2),
            avg_monthly_charge DECIMAL(12,2),
            avg_satisfaction DECIMAL(8,2),
            churn_rate_pct DECIMAL(8,2),
            revenue_at_risk DECIMAL(16,2),
            share_of_customers_pct DECIMAL(8,2),
            PRIMARY KEY (state)
        ) ENGINE=InnoDB;
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS city_retention_profile (
            state VARCHAR(100),
            city VARCHAR(150),
            customers INT NOT NULL,
            churned_customers INT NOT NULL,
            avg_cltv DECIMAL(14,2),
            total_revenue DECIMAL(16,2),
            avg_monthly_charge DECIMAL(12,2),
            avg_satisfaction DECIMAL(8,2),
            churn_rate_pct DECIMAL(8,2),
            revenue_at_risk DECIMAL(16,2),
            PRIMARY KEY (state, city)
        ) ENGINE=InnoDB;
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS market_segment_risk (
            state VARCHAR(100),
            city VARCHAR(150),
            segment_name VARCHAR(120),
            customers INT NOT NULL,
            churned_customers INT NOT NULL,
            avg_cltv DECIMAL(14,2),
            revenue DECIMAL(16,2),
            churn_rate_pct DECIMAL(8,2),
            revenue_at_risk DECIMAL(16,2)
        ) ENGINE=InnoDB;
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS geographic_grid_profile (
            lat_grid DECIMAL(8,4),
            lon_grid DECIMAL(8,4),
            customers INT NOT NULL,
            churned_customers INT NOT NULL,
            avg_cltv DECIMAL(14,2),
            revenue DECIMAL(16,2),
            churn_rate_pct DECIMAL(8,2),
            revenue_at_risk DECIMAL(16,2),
            PRIMARY KEY (lat_grid, lon_grid)
        ) ENGINE=InnoDB;
    """)

    connection.commit()
    print_result("I created or confirmed all four Phase 6 reporting tables.")
finally:
    cursor.close()
    connection.close()

Result: I created or confirmed all four Phase 6 reporting tables.


### Result & conclusion

The MySQL reporting layer now has dedicated tables for state retention, city retention, market-segment risk, and geographic grid risk.

## 2. Publish state retention profiles

In [6]:
connection = get_connection()
cursor = connection.cursor()

try:
    cursor.execute("DELETE FROM state_retention_profile")
    sql = """
        INSERT INTO state_retention_profile (
            state, customers, churned_customers, avg_cltv,
            total_revenue, avg_monthly_charge, avg_satisfaction,
            churn_rate_pct, revenue_at_risk, share_of_customers_pct
        ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """
    rows = []
    for row in state_profile.itertuples(index=False):
        rows.append((
            str(row.state), int(row.customers), int(row.churned_customers),
            float(row.avg_cltv), float(row.total_revenue), float(row.avg_monthly_charge),
            float(row.avg_satisfaction), float(row.churn_rate_pct),
            float(row.revenue_at_risk), float(row.share_of_customers_pct)
        ))
    cursor.executemany(sql, rows)
    connection.commit()
    print_result(f"I published {cursor.rowcount:,} state retention rows.")
finally:
    cursor.close()
    connection.close()

Result: I published 1 state retention rows.


### Result & conclusion

State retention is now available as a persistent SQL table for reporting and dashboarding.

## 3. Publish city retention profiles

In [ ]:
connection = get_connection()
cursor = connection.cursor()

try:
    cursor.execute("DELETE FROM city_retention_profile")
    sql = """
        INSERT INTO city_retention_profile (
            state, city, customers, churned_customers, avg_cltv,
            total_revenue, avg_monthly_charge, avg_satisfaction,
            churn_rate_pct, revenue_at_risk
        ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """
    rows = []
    for row in city_profile.itertuples(index=False):
        rows.append((
            None if pd.isna(row.state) else str(row.state),
            None if pd.isna(row.city) else str(row.city),
            int(row.customers), int(row.churned_customers), float(row.avg_cltv),
            float(row.total_revenue), float(row.avg_monthly_charge),
            float(row.avg_satisfaction), float(row.churn_rate_pct),
            float(row.revenue_at_risk)
        ))
    cursor.executemany(sql, rows)
    connection.commit()
    print_result(f"I published {cursor.rowcount:,} city retention rows.")
finally:
    cursor.close()
    connection.close()

Result: I published 1,106 city retention rows.


### Result & conclusion

City-level retention is now available for detailed market analysis. The composite key `(state, city)` preserves the geography hierarchy used in the notebook.

## 4. Publish market × segment risk

In [8]:
connection = get_connection()
cursor = connection.cursor()

try:
    cursor.execute("DELETE FROM market_segment_risk")
    sql = """
        INSERT INTO market_segment_risk (
            state, city, segment_name, customers, churned_customers,
            avg_cltv, revenue, churn_rate_pct, revenue_at_risk
        ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """
    rows = []
    for row in market_segment_risk.itertuples(index=False):
        rows.append((
            None if pd.isna(row.state) else str(row.state),
            None if pd.isna(row.city) else str(row.city),
            None if pd.isna(row.segment_name) else str(row.segment_name),
            int(row.customers), int(row.churned_customers), float(row.avg_cltv),
            float(row.revenue), float(row.churn_rate_pct), float(row.revenue_at_risk)
        ))
    cursor.executemany(sql, rows)
    connection.commit()
    print_result(f"I published {cursor.rowcount:,} market-segment risk rows.")
finally:
    cursor.close()
    connection.close()

Result: I published 61 market-segment risk rows.


### Result & conclusion

The segment-market risk layer is now queryable in MySQL. This is the main Phase 6 bridge into targeted retention strategy analysis.

## 5. Publish the geographic grid profile

In [9]:
connection = get_connection()
cursor = connection.cursor()

try:
    cursor.execute("DELETE FROM geographic_grid_profile")
    sql = """
        INSERT INTO geographic_grid_profile (
            lat_grid, lon_grid, customers, churned_customers,
            avg_cltv, revenue, churn_rate_pct, revenue_at_risk
        ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
    """
    rows = []
    for row in grid_profile.itertuples(index=False):
        rows.append((
            float(row.lat_grid), float(row.lon_grid), int(row.customers),
            int(row.churned_customers), float(row.avg_cltv), float(row.revenue),
            float(row.churn_rate_pct), float(row.revenue_at_risk)
        ))
    cursor.executemany(sql, rows)
    connection.commit()
    print_result(f"I published {cursor.rowcount:,} geographic grid rows.")
finally:
    cursor.close()
    connection.close()

Result: I published 1,624 geographic grid rows.


### Result & conclusion

The geographic grid is now stored as a reusable SQL table. It supports spatial reporting without needing to recompute the coordinate aggregation each time.

## 6. Create reusable reporting views

In [10]:
connection = get_connection()
cursor = connection.cursor()

try:
    cursor.execute("""
        CREATE OR REPLACE VIEW vw_market_retention AS
        SELECT
            state, city, customers, churned_customers,
            churn_rate_pct, avg_cltv, total_revenue,
            revenue_at_risk, avg_satisfaction, avg_monthly_charge
        FROM city_retention_profile;
    """)

    cursor.execute("""
        CREATE OR REPLACE VIEW vw_market_segment_risk AS
        SELECT
            state, city, segment_name, customers,
            churned_customers, churn_rate_pct, avg_cltv,
            revenue, revenue_at_risk
        FROM market_segment_risk;
    """)

    cursor.execute("""
        CREATE OR REPLACE VIEW vw_geographic_risk_grid AS
        SELECT
            lat_grid, lon_grid, customers,
            churned_customers, churn_rate_pct,
            avg_cltv, revenue, revenue_at_risk
        FROM geographic_grid_profile;
    """)

    connection.commit()
    print_result("I created three reporting views for city retention, market-segment risk, and geographic grid risk.")
finally:
    cursor.close()
    connection.close()

Result: I created three reporting views for city retention, market-segment risk, and geographic grid risk.


### Result & conclusion

The reporting views provide stable SQL entry points for later analysis tools. I can use them directly in Power BI or SQL without repeating notebook transformations.

## 7. Validate the published layer

In [11]:
connection = get_connection()
cursor = connection.cursor(dictionary=True)

try:
    cursor.execute("""
        SELECT
            (SELECT COUNT(*) FROM fact_customer_status) AS source_customers,
            (SELECT COUNT(*) FROM state_retention_profile) AS state_rows,
            (SELECT COUNT(*) FROM city_retention_profile) AS city_rows,
            (SELECT COUNT(*) FROM market_segment_risk) AS market_segment_rows,
            (SELECT COUNT(*) FROM geographic_grid_profile) AS grid_rows;
    """)
    validation = pd.DataFrame(cursor.fetchall())
finally:
    cursor.close()
    connection.close()

display(validation)
source_customers = int(validation.loc[0, "source_customers"])
print_result(
    f"The published layer contains {int(validation.loc[0, 'state_rows']):,} state rows, {int(validation.loc[0, 'city_rows']):,} city rows, {int(validation.loc[0, 'market_segment_rows']):,} market-segment rows, and {int(validation.loc[0, 'grid_rows']):,} grid rows against {source_customers:,} source customers."
)

,source_customers,state_rows,city_rows,market_segment_rows,grid_rows
0,7043,1,1106,61,1624


Result: The published layer contains 1 state rows, 1,106 city rows, 61 market-segment rows, and 1,624 grid rows against 7,043 source customers.


### Result & conclusion

The row-count check confirms that the reporting objects exist and contain data. The city, market-segment, and grid counts are expected to differ because they are aggregated grains rather than customer-level tables.

## 8. Final export of reporting artifacts

In [12]:
output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

state_profile.to_csv(output_dir / "state_retention_profile.csv", index=False)
city_profile.to_csv(output_dir / "city_retention_profile.csv", index=False)
market_segment_risk.to_csv(output_dir / "market_segment_risk.csv", index=False)
grid_profile.to_csv(output_dir / "geographic_grid_profile.csv", index=False)

print_result("I kept the final Phase 6 CSV artifacts alongside the MySQL reporting layer so the analysis remains portable.")

Result: I kept the final Phase 6 CSV artifacts alongside the MySQL reporting layer so the analysis remains portable.


### Result & conclusion

Phase 6 now has both portable CSV outputs and a persistent MySQL reporting layer. This makes the work easier to reuse in dashboards, SQL analysis, and the upcoming retention strategy phase.

# Phase 6 final conclusion

I have moved from geographic data validation to market-level retention analysis, segment geography, and a reusable MySQL reporting layer. The next phase can use these outputs to translate the observed retention patterns into business actions and BI reporting.